# Resume Parser Evaluation v2 — Larger Labeled Benchmark

This notebook evaluates **semantic-only**, **TF-IDF-only**, and **hybrid** ranking across 12 fictional resume profiles and 8 job descriptions (96 labeled resume–job pairs).

**Ground truth:** The relevance labels are hand-authored illustrative labels for fictional profiles. This dataset is useful for checking the evaluation workflow, not for claiming real-world model performance. Replace or validate labels with human reviewers before using results in a report.

## 1. Install dependencies

In [1]:
%pip install -q sentence-transformers scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 2. Imports and load dataset

In [2]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

DATA_PATH = Path("labeled_resume_job_pairs.csv")
if not DATA_PATH.exists():
    # If the CSV is stored next to this notebook, this path should resolve.
    raise FileNotFoundError("Place labeled_resume_job_pairs.csv in the same folder as this notebook.")

data = pd.read_csv(DATA_PATH)
print("Rows:", len(data))
print("Resume profiles:", data["resume_id"].nunique())
print("Job descriptions:", data["role"].nunique())
data.head()

Rows: 96
Resume profiles: 12
Job descriptions: 8


,resume_id,profile_focus,role,relevance,resume_text,job_description
0,R01,ML Engineer,ML Engineer,2,"Computer science graduate with Python, scikit-...","Develop, train, evaluate, and deploy machine l..."
1,R01,ML Engineer,Data Analyst,1,"Computer science graduate with Python, scikit-...","Analyze business datasets with SQL, Excel, Pyt..."
2,R01,ML Engineer,Backend Python Developer,1,"Computer science graduate with Python, scikit-...","Build backend services using Python, FastAPI o..."
3,R01,ML Engineer,Frontend Developer,0,"Computer science graduate with Python, scikit-...",Create responsive web interfaces using JavaScr...
4,R01,ML Engineer,DevOps Engineer,0,"Computer science graduate with Python, scikit-...",Automate infrastructure and releases using Lin...


## 3. Load semantic model and create ranking scores

In [3]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

def score_semantic(resume_text, job_texts):
    embeddings = model.encode(
        [resume_text] + list(job_texts),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )
    return embeddings[1:] @ embeddings[0]

def score_tfidf(resume_text, job_texts):
    texts = [resume_text] + list(job_texts)
    matrix = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        sublinear_tf=True
    ).fit_transform(texts)
    return cosine_similarity(matrix[0:1], matrix[1:]).ravel()

ranking_rows = []
for resume_id, group in data.groupby("resume_id", sort=True):
    # Each resume is repeated across jobs; use one resume text per resume_id.
    resume_text = group["resume_text"].iloc[0]
    jobs_for_resume = group[["role", "job_description", "relevance"]].drop_duplicates("role")
    roles = jobs_for_resume["role"].tolist()
    job_texts = jobs_for_resume["job_description"].tolist()

    semantic = score_semantic(resume_text, job_texts)
    lexical = score_tfidf(resume_text, job_texts)
    hybrid = 0.7 * semantic + 0.3 * lexical

    for i, (_, row) in enumerate(jobs_for_resume.iterrows()):
        ranking_rows.append({
            "resume_id": resume_id,
            "profile_focus": group["profile_focus"].iloc[0],
            "role": roles[i],
            "relevance": int(row["relevance"]),
            "semantic_score": float(semantic[i]),
            "tfidf_score": float(lexical[i]),
            "hybrid_score": float(hybrid[i]),
        })

rankings = pd.DataFrame(ranking_rows)
rankings.head()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,resume_id,profile_focus,role,relevance,semantic_score,tfidf_score,hybrid_score
0,R01,ML Engineer,ML Engineer,2,0.596759,0.220607,0.483913
1,R01,ML Engineer,Data Analyst,1,0.419680,0.027851,0.302131
2,R01,ML Engineer,Backend Python Developer,1,0.315598,0.030016,0.229923
3,R01,ML Engineer,Frontend Developer,0,0.100172,0.000000,0.070121
4,R01,ML Engineer,DevOps Engineer,0,0.270310,0.000000,0.189217


## 4. Ranking metrics

- Precision@K / Recall@K treat relevance labels greater than 0 as relevant.
- nDCG@K uses graded relevance (0, 1, 2).
- MRR measures how high the first relevant result appears.

Because the ranking task is one resume against a set of job descriptions, recall uses the number of relevant jobs among the 8 roles for that resume as its denominator.

In [4]:
def precision_at_k(relevances, k):
    relevances = np.asarray(relevances)
    return float(np.sum(relevances[:k] > 0) / k)

def recall_at_k(relevances, k):
    relevances = np.asarray(relevances)
    total_relevant = np.sum(relevances > 0)
    return float(np.sum(relevances[:k] > 0) / total_relevant) if total_relevant else np.nan

def dcg_at_k(relevances, k):
    rel = np.asarray(relevances[:k], dtype=float)
    if not len(rel):
        return 0.0
    return float(np.sum((2 ** rel - 1) / np.log2(np.arange(2, len(rel) + 2))))

def ndcg_at_k(relevances, k):
    actual = dcg_at_k(relevances, k)
    ideal = dcg_at_k(sorted(relevances, reverse=True), k)
    return actual / ideal if ideal else np.nan

def mrr(relevances):
    for rank, rel in enumerate(relevances, start=1):
        if rel > 0:
            return 1 / rank
    return 0.0

def evaluate_method(rankings, score_col, k=3):
    per_resume = []
    for resume_id, group in rankings.groupby("resume_id"):
        ordered = group.sort_values(score_col, ascending=False)
        rels = ordered["relevance"].tolist()
        per_resume.append({
            "resume_id": resume_id,
            "method": score_col,
            "Precision@1": precision_at_k(rels, 1),
            f"Recall@{k}": recall_at_k(rels, k),
            f"nDCG@{k}": ndcg_at_k(rels, k),
            "MRR": mrr(rels),
            "top_role": ordered.iloc[0]["role"],
            "top_relevance": int(ordered.iloc[0]["relevance"]),
        })
    return pd.DataFrame(per_resume)

metric_frames = [
    evaluate_method(rankings, "semantic_score", k=3),
    evaluate_method(rankings, "tfidf_score", k=3),
    evaluate_method(rankings, "hybrid_score", k=3),
]
per_resume_metrics = pd.concat(metric_frames, ignore_index=True)
metric_columns = ["Precision@1", "Recall@3", "nDCG@3", "MRR"]
comparison = per_resume_metrics.groupby("method")[metric_columns].mean().sort_values("nDCG@3", ascending=False)
display(comparison)

,Precision@1,Recall@3,nDCG@3,MRR
method,,,,
tfidf_score,1.000000,0.780556,0.960953,1.000000
hybrid_score,0.916667,0.808333,0.916667,0.933333
semantic_score,0.916667,0.808333,0.916667,0.933333


## 5. Inspect top-ranked predictions

In [5]:
top_predictions = per_resume_metrics[[
    "resume_id", "method", "top_role", "top_relevance",
    "Precision@1", "Recall@3", "nDCG@3", "MRR"
]].sort_values(["resume_id", "method"])
display(top_predictions)

,resume_id,method,top_role,top_relevance,Precision@1,Recall@3,nDCG@3,MRR
24,R01,hybrid_score,ML Engineer,2,1.0,0.600000,1.000000,1.0
0,R01,semantic_score,ML Engineer,2,1.0,0.600000,1.000000,1.0
12,R01,tfidf_score,ML Engineer,2,1.0,0.600000,1.000000,1.0
25,R02,hybrid_score,Data Analyst,2,1.0,1.000000,1.000000,1.0
1,R02,semantic_score,Data Analyst,2,1.0,1.000000,1.000000,1.0
13,R02,tfidf_score,Data Analyst,2,1.0,1.000000,1.000000,1.0
26,R03,hybrid_score,Backend Python Developer,2,1.0,0.500000,1.000000,1.0
2,R03,semantic_score,Backend Python Developer,2,1.0,0.500000,1.000000,1.0
14,R03,tfidf_score,Backend Python Developer,2,1.0,0.500000,1.000000,1.0
27,R04,hybrid_score,Frontend Developer,2,1.0,1.000000,1.000000,1.0


## 6. Inspect where methods disagree

In [6]:
top_role_wide = per_resume_metrics.pivot(index="resume_id", columns="method", values="top_role")
display(top_role_wide)
disagreements = top_role_wide[top_role_wide.nunique(axis=1) > 1]
print("Profiles where top-ranked role differs by method:", len(disagreements))
display(disagreements)

method,hybrid_score,semantic_score,tfidf_score
resume_id,,,
R01,ML Engineer,ML Engineer,ML Engineer
R02,Data Analyst,Data Analyst,Data Analyst
R03,Backend Python Developer,Backend Python Developer,Backend Python Developer
R04,Frontend Developer,Frontend Developer,Frontend Developer
R05,DevOps Engineer,DevOps Engineer,DevOps Engineer
R06,NLP Engineer,NLP Engineer,NLP Engineer
R07,Computer Vision Engineer,Computer Vision Engineer,Computer Vision Engineer
R08,Database Developer,Database Developer,Database Developer
R09,Data Analyst,Data Analyst,Data Analyst


Profiles where top-ranked role differs by method: 1


method,hybrid_score,semantic_score,tfidf_score
resume_id,,,
R10,Data Analyst,Data Analyst,DevOps Engineer


## 7. Export metrics and rankings

In [7]:
rankings.to_csv("resume_job_ranking_evaluation_v2.csv", index=False)
per_resume_metrics.to_csv("resume_ranking_metrics_by_resume_v2.csv", index=False)
comparison.to_csv("resume_ranking_metric_comparison_v2.csv")
print("Exported v2 ranking and metric CSV files.")

Exported v2 ranking and metric CSV files.


## 8. Interpretation and limitations

- Do not choose a winner solely from this small synthetic benchmark.
- The profiles and relevance labels are fictional and hand-authored; they are not validated by hiring experts.
- The benchmark deliberately includes partial and negative matches to make ranking differences easier to inspect.
- Use a human-reviewed, held-out dataset before reporting model performance.
- Do not interpret similarity scores as hiring probabilities or use them as the sole basis for hiring decisions.